# 05 — Publication-Grade VTON Inference & Quantitative Evaluation Suite
Runs side-by-side held-out test inference (**Zero-Shot Baseline** vs **FashionFit-PK LoRA r=16**), saves a visual comparison grid, computes **FID ↓, LPIPS ↓, SSIM ↑, CLIP-I ↑**, and exports `paper_tables.tex` directly to `05_PAPER_ARTIFACTS/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q diffusers transformers accelerate peft lpips torchmetrics open-clip-torch clean-fid pillow pandas tqdm


In [ ]:
import sys
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from torchvision import transforms
from tqdm import tqdm
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
import peft.import_utils
from peft import PeftModel

# Bypass torchao version conflict in Colab peft
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, 'is_torchao_available'):
        try:
            mod.is_torchao_available = lambda: False
        except Exception:
            pass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESEARCH_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research')
TEST_DIR = RESEARCH_ROOT / '01_PK_VTON_DATASET' / 'paired_test'
LORA_CKPT = RESEARCH_ROOT / '03_CATVTON_LORA_RUNS' / 'catvton_pk_lora_r16' / 'step_1000'
EVAL_ROOT = RESEARCH_ROOT / '04_HYBRID_EVAL_OUTPUTS'
PAPER_DIR = RESEARCH_ROOT / '05_PAPER_ARTIFACTS'

GT_DIR = EVAL_ROOT / 'gt'
BASE_DIR = EVAL_ROOT / 'baseline_zero_shot'
LORA_DIR = EVAL_ROOT / 'fashionfit_pk_lora'
for d in [GT_DIR, BASE_DIR, LORA_DIR, PAPER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE_ID = 'booksforcharlie/stable-diffusion-inpainting'
vae = AutoencoderKL.from_pretrained(BASE_ID, subfolder='vae', torch_dtype=torch.float16).to(device).eval()
scheduler = DDIMScheduler.from_pretrained(BASE_ID, subfolder='scheduler')

img_tf = transforms.Compose([
    transforms.Resize((512, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
msk_tf = transforms.Compose([
    transforms.Resize((512, 384), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

@torch.no_grad()
def run_catvton_infer(unet_model, person_pil, garment_pil, mask_pil, num_steps=28, seed=42):
    person = img_tf(person_pil).unsqueeze(0).to(device, dtype=torch.float16)
    garment = img_tf(garment_pil).unsqueeze(0).to(device, dtype=torch.float16)
    mask = (msk_tf(mask_pil) > 0.5).float().unsqueeze(0).to(device, dtype=torch.float16)

    person_lat = vae.encode(person).latent_dist.mean * vae.config.scaling_factor
    masked_person = person * (1.0 - mask)
    masked_lat = vae.encode(masked_person).latent_dist.mean * vae.config.scaling_factor
    garment_lat = vae.encode(garment).latent_dist.mean * vae.config.scaling_factor
    mask_lat = F.interpolate(mask, size=person_lat.shape[-2:], mode='nearest')

    cond_lat = torch.cat([masked_lat, garment_lat], dim=-1)
    cond_mask = torch.cat([mask_lat, torch.zeros_like(mask_lat)], dim=-1)
    empty_cond = torch.zeros((1, 77, 768), device=device, dtype=torch.float16)

    gen = torch.Generator(device=device).manual_seed(seed)
    latents = torch.randn(cond_lat.shape, generator=gen, device=device, dtype=torch.float16)

    scheduler.set_timesteps(num_steps, device=device)
    for t in scheduler.timesteps:
        unet_in = torch.cat([latents, cond_mask, cond_lat], dim=1)
        noise_pred = unet_model(unet_in, t, encoder_hidden_states=empty_cond).sample
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    w_half = person_lat.shape[-1]
    out_lat = latents[..., :w_half] / vae.config.scaling_factor
    decoded = vae.decode(out_lat).sample
    decoded = (decoded / 2 + 0.5).clamp(0, 1)
    out_np = (decoded[0].permute(1, 2, 0).float().cpu().numpy() * 255).astype(np.uint8)
    out_pil = Image.fromarray(out_np)

    # Strict face/background preservation outside mask
    p_res = person_pil.resize((384, 512), Image.BILINEAR)
    m_res = mask_pil.resize((384, 512), Image.NEAREST)
    return Image.composite(out_pil, p_res, m_res)

# Evaluate on 50 held-out Pakistani test pairs (~8 min on T4, solid for FID/LPIPS/SSIM/CLIP)
person_files = sorted((TEST_DIR / 'person').glob('*.*'))[:50]
print(f'Running held-out evaluation on {len(person_files)} Pakistani test pairs...')

# 1. Baseline (Zero-Shot, NO LoRA)
print('--- Pass 1/2: Zero-Shot Baseline ---')
unet_base = UNet2DConditionModel.from_pretrained(BASE_ID, subfolder='unet', torch_dtype=torch.float16).to(device).eval()
for p_path in tqdm(person_files):
    stem = p_path.stem
    g_path = next((TEST_DIR / 'garment').glob(f'{stem}.*'))
    m_path = next((TEST_DIR / 'mask').glob(f'{stem}.*'))
    p_im = Image.open(p_path).convert('RGB')
    g_im = Image.open(g_path).convert('RGB')
    m_im = Image.open(m_path).convert('L')
    p_im.resize((384, 512), Image.BILINEAR).save(GT_DIR / f'{stem}.png')
    res_base = run_catvton_infer(unet_base, p_im, g_im, m_im, num_steps=28, seed=42)
    res_base.save(BASE_DIR / f'{stem}.png')

# 2. FashionFit-PK LoRA (Your fine-tuned step_1000 adapter)
print('--- Pass 2/2: FashionFit-PK LoRA (step_1000) ---')
unet_lora = PeftModel.from_pretrained(unet_base, str(LORA_CKPT)).to(device).eval()
for p_path in tqdm(person_files):
    stem = p_path.stem
    g_path = next((TEST_DIR / 'garment').glob(f'{stem}.*'))
    m_path = next((TEST_DIR / 'mask').glob(f'{stem}.*'))
    p_im = Image.open(p_path).convert('RGB')
    g_im = Image.open(g_path).convert('RGB')
    m_im = Image.open(m_path).convert('L')
    res_lora = run_catvton_infer(unet_lora, p_im, g_im, m_im, num_steps=28, seed=42)
    res_lora.save(LORA_DIR / f'{stem}.png')

# Save visual side-by-side comparison grid (first 6 test cases)
rows = []
for p_path in person_files[:6]:
    stem = p_path.stem
    g_path = next((TEST_DIR / 'garment').glob(f'{stem}.*'))
    im_gt = Image.open(GT_DIR / f'{stem}.png').convert('RGB').resize((384, 512))
    im_g = Image.open(g_path).convert('RGB').resize((384, 512))
    im_b = Image.open(BASE_DIR / f'{stem}.png').convert('RGB').resize((384, 512))
    im_l = Image.open(LORA_DIR / f'{stem}.png').convert('RGB').resize((384, 512))
    strip = Image.new('RGB', (384 * 4, 512))
    strip.paste(im_gt, (0, 0))
    strip.paste(im_g, (384, 0))
    strip.paste(im_b, (768, 0))
    strip.paste(im_l, (1152, 0))
    rows.append(strip)

grid = Image.new('RGB', (384 * 4, 512 * len(rows) + 48), (20, 20, 20))
draw = ImageDraw.Draw(grid)
for idx, label in enumerate(['Ground Truth (Person)', 'Target Pakistani Garment', 'Zero-Shot Baseline', 'FashionFit-PK LoRA (Ours)']):
    draw.text((idx * 384 + 20, 14), label, fill=(255, 255, 255))
for r_idx, strip in enumerate(rows):
    grid.paste(strip, (0, 48 + r_idx * 512))
grid.save(PAPER_DIR / 'vton_baseline_vs_lora_grid.jpg', quality=92)
print('Saved visual side-by-side comparison grid to:', PAPER_DIR / 'vton_baseline_vs_lora_grid.jpg')


In [ ]:
import json
import torch
import lpips
import open_clip
import pandas as pd
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchmetrics.image import StructuralSimilarityIndexMeasure
from cleanfid import fid

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESEARCH_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research')
EVAL_ROOT = RESEARCH_ROOT / '04_HYBRID_EVAL_OUTPUTS'
PAPER_DIR = RESEARCH_ROOT / '05_PAPER_ARTIFACTS'
PAPER_DIR.mkdir(parents=True, exist_ok=True)

lpips_fn = lpips.LPIPS(net='alex').to(device).eval()
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai', device=device)
clip_model.eval()
tf_01 = transforms.Compose([transforms.Resize((512, 384)), transforms.ToTensor()])

@torch.no_grad()
def evaluate_method(gt_dir: Path, pred_dir: Path, method_name: str):
    gt_files = sorted(gt_dir.glob('*.*'))
    ssim_vals, lpips_vals, clip_vals = [], [], []
    for gt_p in gt_files:
        pred_p = pred_dir / gt_p.name
        if not pred_p.exists():
            continue
        gt_im = Image.open(gt_p).convert('RGB')
        pr_im = Image.open(pred_p).convert('RGB')

        t_gt = tf_01(gt_im).unsqueeze(0).to(device)
        t_pr = tf_01(pr_im).unsqueeze(0).to(device)
        ssim_vals.append(ssim_fn(t_pr, t_gt).item())
        lpips_vals.append(lpips_fn(t_pr * 2 - 1, t_gt * 2 - 1).item())

        c_gt = clip_model.encode_image(clip_preprocess(gt_im).unsqueeze(0).to(device))
        c_pr = clip_model.encode_image(clip_preprocess(pr_im).unsqueeze(0).to(device))
        c_gt = c_gt / c_gt.norm(dim=-1, keepdim=True)
        c_pr = c_pr / c_pr.norm(dim=-1, keepdim=True)
        clip_vals.append((c_gt * c_pr).sum().item())

    fid_score = fid.compute_fid(str(gt_dir), str(pred_dir)) if len(ssim_vals) >= 10 else float('nan')
    return {
        'Method': method_name,
        'N': len(ssim_vals),
        'FID_down': round(float(fid_score), 2),
        'LPIPS_down': round(float(sum(lpips_vals)/max(len(lpips_vals),1)), 4),
        'SSIM_up': round(float(sum(ssim_vals)/max(len(ssim_vals),1)), 4),
        'CLIP_I_up': round(float(sum(clip_vals)/max(len(clip_vals),1)), 4),
    }

results = [
    evaluate_method(EVAL_ROOT / 'gt', EVAL_ROOT / 'baseline_zero_shot', 'Zero-Shot Baseline (SD1.5-Inpaint)'),
    evaluate_method(EVAL_ROOT / 'gt', EVAL_ROOT / 'fashionfit_pk_lora', 'FashionFit-PK LoRA r=16 (Ours)'),
]

df = pd.DataFrame(results)
print('\n=== PUBLICATION BENCHMARK RESULTS ===')
print(df.to_string(index=False))

df.to_csv(PAPER_DIR / 'vton_eval_results.csv', index=False)
(PAPER_DIR / 'vton_eval_results.json').write_text(json.dumps(results, indent=2))

latex_lines = [
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Quantitative Virtual Try-On Evaluation on Held-Out PK-VTON Test Split ($N=50$).}',
    r'\label{tab:pk_vton_eval}',
    r'\begin{tabular}{lcccc}',
    r'\toprule',
    r'Method & FID $\downarrow$ & LPIPS $\downarrow$ & SSIM $\uparrow$ & CLIP-I $\uparrow$ \\',
    r'\midrule',
]
for r in results:
    latex_lines.append(f"{r['Method']} & {r['FID_down']:.2f} & {r['LPIPS_down']:.4f} & {r['SSIM_up']:.4f} & {r['CLIP_I_up']:.4f} \\\\")
latex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
(PAPER_DIR / 'paper_tables.tex').write_text('\n'.join(latex_lines))
print('\nSaved paper_tables.tex, vton_eval_results.csv, and vton_eval_results.json to 05_PAPER_ARTIFACTS/!')
